In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Loading Dataset file

In [ ]:
data = pd.read_csv("/kaggle/input/smart-agricultural-production-optimizing-engine/Crop_recommendation.csv")

# Finding the Shape of Dataset

In [ ]:
data.shape

# Inspecting the first 10 rows of the Dataset

In [ ]:
data.head(10)

# Checking the Null values if any?

In [ ]:
data.isnull().sum()

# Checking the Values

In [ ]:
data["label"].value_counts()

# Finding the Average of different Variables

In [ ]:
for col in data.columns:
    if col == 'label': continue
    print("Average", col, "required: {:.2f}".format(data[col].mean()))

# Summary statitistics of each crop

In [ ]:
@interact
def summary(crops = list(data['label'].value_counts().index)):
    x = data[data['label'] == crops]
    for col in data.columns:
        if col == 'label':
            continue
        cur = ""
        if col == 'N': 
            cur = "Nitrogen"
        elif col == 'P':
            cur = 'Phosphorus'
        elif col == 'K':
            cur = 'Potassium'
        else:
            cur = col
        print("---------------------------------------------")
        print("Statistics for", cur)
        print("Minimum", cur, "required: {:.2f}".format(x[str(col)].min()))
        print("Average", cur, "required: {:.2f}".format(x[str(col)].mean()))
        print("Maximum", cur, "required: {:.2f}".format(x[str(col)].max()))

# Comparison of the average requirement for each crop with average conditions


In [ ]:
@interact
def compare(conditions = ['N','P','K','temperature','ph','humidity','rainfall']):
    print("Average Value for", conditions, "is {0:.2f}".format(data[conditions].mean()))
    print("----------------------------------------------")
    for label in data.label.unique():
        print(label, ": {0:.2f}".format(data[(data['label'] == label)][conditions].mean()))

#  Making this function more Intuitive

In [ ]:
@interact
def compare(conditions = ['N','P','K','temperature','ph','humidity','rainfall']):
    print("Crops which require greater than average", conditions, '\n')
    print(data[data[conditions] > data[conditions].mean()]['label'].unique())
    print("-------------------------------------")
    print("Crops which require less than average", conditions,'\n')
    print(data[data[conditions] <=data[conditions].mean()]['label'].unique())

# Distributions for Agricultural Conditions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create subplots with 2 rows and 4 columns
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Define the data columns and colors
columns = ['N', 'P', 'K', 'temperature', 'rainfall', 'humidity', 'ph']
colors = ['lightgrey', 'lightblue', 'darkblue', 'black', 'grey', 'lightgreen', 'darkgreen']

# Loop through the subplots and create the distribution plots
for i, col in enumerate(columns):
    row = i // 4
    col_num = i % 4
    
    sns.histplot(data[col], color=colors[i], ax=axes[row, col_num], kde=True)
    axes[row, col_num].set_xlabel(f'Ratio of {col}', fontsize=12)
    axes[row, col_num].grid()

# Adjust the spacing between subplots
plt.subplots_adjust(wspace=0.4, hspace=0.4)

# Add a title for the entire figure
plt.suptitle('Distribution for Agricultural Conditions', fontsize=20)

# Show the plot
plt.show()

# Finding out Some Interesting Facts.

In [ ]:
print("Crops which requires very High Ratio of Nirtogen Content in Soil:", data[data['N'] > 120]['label'].unique())
print("Crops which requires very High Ratio of Phosphorous Content in Soil:", data[data['P'] > 100]['label'].unique())
print("Crops which requires very High Ratio of Potassium Content in Soil:", data[data['K'] > 200]['label'].unique())
print("Crops which requires very High Rainfall:", data[data['rainfall'] > 200]['label'].unique())
print("Crops which requires very Low Temperature:", data[data['temperature'] > 10]['label'].unique())
print("Crops which requires very High Temperature:", data[data['temperature'] > 40]['label'].unique())
print("Crops which requires very Low Humidty:", data[data['humidity'] > 20]['label'].unique())
print("Crops which requires very Low pH:", data[data['ph'] < 4]['label'].unique())
print("Cr0ps which requires very Low pH:", data[data['ph'] > 9]['label'].unique())

# Seasonal crops recommendations (Summer, Winter, Rainy)

In [ ]:
print("Summer Crops")
print(data[(data['temperature'] > 30) & (data['humidity'] > 50)]['label'].unique())
print("-----------------------------------------")
print("Winter Crops")
print(data[(data['temperature'] < 20) & (data['humidity'] > 30)]['label'].unique())
print("-----------------------------------------")
print("Rainy Crops")
print(data[(data['rainfall'] > 200) & (data['humidity'] > 30)]['label'].unique())

# Removing of columns to find some more Insights.

In [ ]:
from sklearn.cluster import KMeans

# removing the labels column
x = data.drop(['label'], axis=1)

# selecting all the values of the data
x = x.values

# checking the shape
print(x.shape)

#  To determine the Optimum Number of Clusters within the Dataset

In [ ]:
# Lets determine the Optimum Number of Clusters within the Dataset

plt.rcParams['figure.figsize'] = (10, 4)

wcss = []
for i in range(1, 8):
    km = KMeans(n_clusters = i, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
    km.fit(x)
    wcss.append(km.inertia_)
    
# Lets plot the results
plt.plot(range(1, 8), wcss)
plt.title('The Elbow Method', fontsize = 20)
plt.xlabel('No. of Clusters')
plt.ylabel('wcss')
plt.show()

# Implementation of K Means to perform Clustering Analysis.

In [ ]:
km = KMeans(n_clusters = 4, init = 'k-means++', max_iter = 500, n_init = 20, random_state = 42)
y_means = km.fit_predict(x)
a = data['label']
y_means = pd.DataFrame(y_means)
z = pd.concat([y_means, a], axis = 1)
z = z.rename(columns = {0: 'cluster'})

# Clusters of each crop
print("Crops in First cluster:", z[z['cluster'] == 0][ 'label'].unique())
print("----------------------------------------------------------------")
print("Crops in Second cluster:", z[z['cluster'] == 1][ 'label'].unique())
print("----------------------------------------------------------------")
print("Crops in Third cluster:", z[z['cluster'] == 2][ 'label'].unique())
print("----------------------------------------------------------------")
print("Crops in Forth cluster:", z[z['cluster'] == 3][ 'label'].unique())

# Splitting up of the Dataset for Predictive Modelling

In [ ]:
# Lets split the Dataset for Predictive Modelling
y = data['label']
x = data.drop(['label'], axis = 1)

x.shape, y.shape

# Creations of the Training and Testing Sets for Validation of Results

In [ ]:
# Lets create Training and Testing Sets for Validation of Results
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)

x_train.shape, x_test.shape, y_train.shape, x_test.shape

# Creation of a Predictive Model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

 # Evaluation of the Model Performance

In [ ]:
# Lets evaluate the Model Performance
from sklearn.metrics import confusion_matrix

# Lets print the Confusion matrix first
plt.rcParams['figure.figsize'] = (10, 10)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True,)
plt.title('Confusion Matrix for Logistic Regression', fontsize=15)
plt.show()

# Printing out the Classification Report.

In [ ]:
# Lets print the Classification Repost also
from sklearn.metrics import classification_report

cr = classification_report(y_test, y_pred)
print(cr)

# Inspecting the Head of the Dataset to choose similar sample

In [ ]:
# Lets check the Head of the Dataset
data.head()

# Pridiction

In [ ]:
x = [62,50,41,21,84,8,239]
prediction = model.predict((np.array([x])))
print("The suggested crop for given climatic condition is :", prediction)

## Saving model for deployment

In [ ]:
import pickle
# save the model to disk
filename = 'best_model.pickle'
pickle.dump(model, open(filename, 'wb'))